## 개요
지역검색 데이터(`naver_local_search_mapo.csv`)와 블로그 데이터(`naver_blog_content.csv`)를 place 테이블 스키마에 맞춰 가공.

1. 지역검색 데이터 -> 스키마 매핑
2. 블로그 content에서 장소명/주소 추출 -> 지역검색 API로 재조회해 좌표/카테고리 보강
3. 두 결과 병합 (name+address 기준 중복 제거, 기존 저장분 유지)

In [1]:
import json
import os
import re
import time
import uuid
from datetime import date, datetime, timezone

import dotenv
import pandas as pd
import requests

dotenv.load_dotenv()

LOCAL_SEARCH_URL = "https://naverapihub.apigw.ntruss.com/search/v1/local"
SEARCH_HEADERS = {
    "X-NCP-APIGW-API-KEY-ID": os.getenv("client_ID"),
    "X-NCP-APIGW-API-KEY": os.getenv("client_secret"),
}

PLACE_ID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "nolda.places")


def clean_html(text):
    return re.sub(r"<.*?>", "", text) if isinstance(text, str) else text


def make_place_id(name, address):
    """name+address 기반 결정적 uuid5 -> 재실행해도 같은 장소는 같은 id 유지 (Supabase upsert 대비)"""
    return str(uuid.uuid5(PLACE_ID_NAMESPACE, f"{name}|{address}"))


def to_place_row(item, dong=None, source="local"):
    """지역검색 API 응답 item(dict) -> place 스키마 행. event_start/end는 상시 영업 장소라 항상 비움
    source="blog_popup"인 경우, 카테고리가 비었거나 "팝업스토어"로 시작하지 않으면(엉뚱한 업체와 매칭된 경우 포함)
    "팝업스토어"로 강제 지정 -> 이 파이프라인을 거친 건 전부 팝업이라는 사실은 항상 보장
    """
    lat = item.get("mapy")
    lng = item.get("mapx")
    address = item.get("address")
    name = clean_html(item.get("title"))
    category = item.get("category")
    if source == "blog_popup" and not (isinstance(category, str) and category.startswith("팝업스토어")):
        category = "팝업스토어"
    return {
        "id": make_place_id(name, address),
        "name": name,
        "category": category,
        "address": address,
        "road_address": item.get("roadAddress"),
        "lat": float(lat) / 1e7 if pd.notna(lat) else None,
        "lng": float(lng) / 1e7 if pd.notna(lng) else None,
        "link": item.get("link") if pd.notna(item.get("link")) else None,
        "phone": item.get("telephone") if pd.notna(item.get("telephone")) else None,
        "business_hours": None,
        "menu": None,
        "tags": [],
        "area": dong,
        "event_start": None,
        "event_end": None,
        "source": source,
        "raw_json": json.dumps(item, ensure_ascii=False, default=str),
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }

## 1. 지역검색 데이터 → 스키마 매핑
`title`→`name`, `mapy/mapx`(1e7 나눔)→`lat`/`lng`, `dong`→`area`, `telephone`→`phone`. 원본은 `raw_json`에 통째로 보관.

In [2]:
from nolda_common import EXCLUDED_CATEGORIES

df_local_raw = pd.read_csv("data/naver_local_search_mapo.csv")

# 예전에 모았던 병원/약국/편의점 등 NOLDA 서비스에 부적합한 카테고리는 최종 산출물에서 제외
# (원본 CSV 자체는 그대로 유지 — 나중에 필요해지면 다시 살릴 수 있게)
before = len(df_local_raw)
df_local_raw = df_local_raw[~df_local_raw["category_keyword"].isin(EXCLUDED_CATEGORIES)].reset_index(drop=True)
print(f"부적합 카테고리 제외: {before}건 -> {len(df_local_raw)}건")

local_rows = [
    to_place_row(row.to_dict(), dong=row.get("dong"), source="local")
    for _, row in df_local_raw.iterrows()
]
df_local_places = pd.DataFrame(local_rows)

print(f"지역검색 -> {len(df_local_places)}건 매핑")
df_local_places.head()

부적합 카테고리 제외: 1201건 -> 1201건
지역검색 -> 1201건 매핑


,id,name,category,address,road_address,lat,lng,link,phone,business_hours,menu,tags,area,event_start,event_end,source,raw_json,fetched_at
0,2fc349fc-7818-58b2-8dbe-c93d58c803a4,슈가코인노래연습장 공덕점,오락시설>노래방,서울특별시 마포구 공덕동 105-159 지하 1층,서울특별시 마포구 마포대로 180 지하 1층,37.550182,126.955507,http://m.facebook.com/sugarcoin1,None,None,None,[],공덕동,None,None,local,"{""title"": ""슈가코인노래연습장 공덕점"", ""link"": ""http://m.f...",2026-09-11T04:37:19.792018+00:00
1,546b5896-cbb6-59d9-8a03-b2e96cc5a778,뮤즈코인노래연습장,오락시설>노래방,서울특별시 마포구 신공덕동 20-18 지하1층,서울특별시 마포구 백범로37길 22 지하1층,37.544066,126.954814,NaN,None,None,None,[],공덕동,None,None,local,"{""title"": ""뮤즈코인노래연습장"", ""link"": NaN, ""category""...",2026-09-11T04:37:19.792318+00:00
2,cdc94e5c-cea2-55fc-92e9-e4fb93385f36,클럽스트라이크 볼링장,"스포츠,오락>볼링장",서울특별시 마포구 도화동 36 4층 클럽스트라이크 볼링장,서울특별시 마포구 마포대로 52 4층 클럽스트라이크 볼링장,37.540328,126.947665,https://blog.naver.com/strikepub,None,None,None,[],공덕동,None,None,local,"{""title"": ""클럽스트라이크 <b>볼링장</b>"", ""link"": ""https...",2026-09-11T04:37:19.792960+00:00
3,931ad374-feb2-5dff-b639-43f7a71319eb,뉴청룡볼링장,"스포츠,오락>볼링장",서울특별시 용산구 갈월동 98-38 청룡빌딩 지하1층,서울특별시 용산구 한강대로 257 청룡빌딩 지하1층,37.541356,126.972639,NaN,None,None,None,[],공덕동,None,None,local,"{""title"": ""뉴청룡<b>볼링장</b>"", ""link"": NaN, ""categ...",2026-09-11T04:37:19.793230+00:00
4,6160b3c7-10c8-5bd8-9a97-76df97e45a61,타겟볼링,"스포츠,오락>볼링장",서울특별시 서대문구 대현동 101-7 혜우빌딩 지하1층,서울특별시 서대문구 신촌역로 10 혜우빌딩 지하1층,37.557589,126.943072,NaN,None,None,None,[],공덕동,None,None,local,"{""title"": ""타겟볼링"", ""link"": NaN, ""category"": ""스포...",2026-09-11T04:37:19.793412+00:00


## 2. 블로그 content에서 장소명/주소/기간 추출 → 지역검색 재조회로 보강
먼저 "연예인이 팝업에 방문했다" 류 연예뉴스성 글은 제외함 (블로거명이 언론/미디어스러운 경우, 기사 바이라인이 있는 경우, 제목에 클릭베이트성 단어가 있는 경우).
나머지는 본문에서 정규식으로 (a) "~팝업스토어" 형태의 장소명 후보, (b) "서울(특별시) 마포구 ..." 주소, (c) "YYYY.MM.DD~MM.DD" 류 진행기간을 뽑음.
장소명 후보는 `kiwipiepy`로 형태소 분석해서 명사류만 남기고 동사/형용사/조사/불용어를 제거해 정제함 (완전한 브랜드명 추출은 아니고, 문장형 필러를 줄이는 수준).
정제된 장소명으로 지역검색 API를 다시 호출해 주소에 "마포"가 포함된 결과가 있으면 좌표/카테고리까지 보강하고,
못 찾으면 추출한 주소만 남기고 좌표는 비워둠 (임시 팝업이라 지역검색 DB에 없는 경우가 많음).
진행기간(`event_start`/`event_end`)은 팝업처럼 기간 한정 장소의 만료 판정에 씀 — 3번 병합 단계에서 종료일 지난 항목을 자동으로 분리.

In [3]:
from kiwipiepy import Kiwi

from nolda_common import MAPO_DONGS, USER_WORDS

ADDRESS_PATTERN = re.compile(r"서울(?:특별시)?\s*마포구[^\n]{0,40}")
NAME_PATTERN = re.compile(r"([\w가-힣]+(?:[ ][\w가-힣]+){0,3}[ ]?팝업(?:[ ]?스토어)?)")
DATE_RANGE_PATTERN = re.compile(
    r"(?P<start>(?:20\d{2}[.\-/]|\d{2}[.\-/])?\d{1,2}[.\-/]\d{1,2}(?:\s*\([^)]*\))?)"
    r"\s*~\s*"
    r"(?P<end>(?:20\d{2}[.\-/]|\d{2}[.\-/])?\d{1,2}[.\-/]\d{1,2}(?:\s*\([^)]*\))?)"
)

# 장소 소개/후기가 아니라 "연예인이 팝업에 방문했다" 류 연예뉴스성 글 걸러내기 위한 패턴
NEWS_BLOGGERNAME_PATTERN = re.compile(r"news|News|NEWS|뉴스|이슈|미디어|BBC", re.IGNORECASE)
NEWS_BYLINE_PATTERN = re.compile(r"\[[^\]=]{1,10}=[^\]]{1,20}\]")  # "[서울=뉴스가디언21]" 같은 기사 바이라인
NEWS_TITLE_PATTERN = re.compile(r"포착|완판|매진|참석|신화|비결|화보|미모|전석|눈도장|깜짝\s?등장")


def is_news_style(bloggername, title, content):
    """연예인 방문 소식 등 장소 정보가 아닌 연예뉴스성 글인지 판별"""
    if NEWS_BLOGGERNAME_PATTERN.search(str(bloggername)):
        return True
    if NEWS_TITLE_PATTERN.search(str(title)):
        return True
    if NEWS_BYLINE_PATTERN.search(str(content)[:300]):
        return True
    return False


kiwi = Kiwi()
for word in USER_WORDS:
    kiwi.add_user_word(word, "NNP")

NOUN_TAGS = {"NNG", "NNP", "SL", "SN", "XR"}
NOUN_STOPWORDS = {
    "내일", "오늘", "어제", "진행", "오픈", "추억", "캐릭터", "후기", "방문",
    "예약", "이벤트", "특전", "일정", "소식", "정보", "여러분", "오전", "오후",
    "수", "것", "때", "안녕", "요약", "돈", "곳", "글", "저", "제", "우리",
}


def clean_name_with_pos(text):
    """형태소 분석 후 명사류(NNG/NNP/SL/SN/XR)만 남기고 조사/어미/동사/형용사/불용어 제거"""
    tokens = kiwi.tokenize(text)
    kept = [t.form for t in tokens if t.tag in NOUN_TAGS and t.form not in NOUN_STOPWORDS]
    return " ".join(kept) if kept else text


def extract_dong(address):
    """주소 텍스트에서 마포구 관심 동 이름을 찾아 area로 사용 (지역검색 dong 파라미터가 없는 블로그 유래 장소용)"""
    if not isinstance(address, str):
        return None
    for dong in MAPO_DONGS:
        if dong in address:
            return dong
    return None


def first_meaningful_lines(text, n_lines=3, min_len=4):
    """본문 앞부분에서 너무 짧은 줄(이모지/공백만 있는 줄 등)은 건너뛰고 의미 있는 줄만 이어붙임"""
    lines = [line.strip() for line in str(text).split("\n")]
    lines = [line for line in lines if len(line) >= min_len]
    return " ".join(lines[:n_lines])


def extract_place_candidates(title, content):
    """장소명은 본문 도입부를 우선 보고(안 잡히면 제목), 정규식으로 후보를 자른 뒤 형태소 분석으로 명사만 남김"""
    name_match = NAME_PATTERN.search(first_meaningful_lines(content))
    if not name_match:
        name_match = NAME_PATTERN.search(str(title))
    raw_name = name_match.group(1).strip() if name_match else str(title)
    candidate_name = clean_name_with_pos(raw_name)

    addr_match = ADDRESS_PATTERN.search(str(content))
    extracted_address = addr_match.group(0).strip() if addr_match else None

    return candidate_name, extracted_address


def _parse_date_token(token, fallback_year=None):
    """'9/10', '09.01', '2026.09.22', '26.08.20(목)' 등을 date로 변환. 연도 없으면 fallback_year 사용"""
    token = re.sub(r"\([^)]*\)", "", token).strip()
    parts = [p for p in re.split(r"[./\-]", token) if p]
    if len(parts) == 3:
        y, m, d = parts
        y = int(y)
        if y < 100:
            y += 2000
    elif len(parts) == 2 and fallback_year is not None:
        y = fallback_year
        m, d = parts
    else:
        return None
    try:
        return date(int(y), int(m), int(d))
    except ValueError:
        return None


def extract_event_period(content, post_year=None):
    """본문에서 진행기간(시작일, 종료일) 추출. 형식이 다양해서 못 찾으면 (None, None) — 이 경우 만료 판정 없이 계속 활성 취급"""
    m = DATE_RANGE_PATTERN.search(str(content))
    if not m:
        return None, None
    start = _parse_date_token(m.group("start"), fallback_year=post_year)
    end = _parse_date_token(m.group("end"), fallback_year=start.year if start else post_year)
    return start, end


def enrich_with_local_search(query):
    """추출한 장소명으로 지역검색 재조회 -> 마포구 주소인 첫 결과 반환, 없으면 None"""
    params = {"query": query, "display": 3, "start": 1, "sort": "comment", "format": "json"}
    resp = requests.get(LOCAL_SEARCH_URL, headers=SEARCH_HEADERS, params=params)
    if resp.status_code != 200:
        return None
    for item in resp.json().get("items", []):
        if "마포" in (item.get("address") or ""):
            return item
    return None

In [4]:
df_blog_raw = pd.read_csv("data/naver_blog_content.csv")

blog_place_rows = []
skipped_news = 0
for _, row in df_blog_raw.iterrows():
    if is_news_style(row.get("bloggername"), row["title"], row["content"]):
        skipped_news += 1
        continue

    candidate_name, extracted_address = extract_place_candidates(row["title"], row["content"])

    postdate = row.get("postdate")
    post_year = int(str(int(postdate))[:4]) if pd.notna(postdate) else None
    event_start, event_end = extract_event_period(row["content"], post_year=post_year)

    enriched = enrich_with_local_search(candidate_name)
    time.sleep(0.2)  # API 호출 제한 방지

    if enriched:
        place = to_place_row(enriched, source="blog_popup")
    else:
        name = clean_html(candidate_name)
        place = {
            "id": make_place_id(name, extracted_address),
            "name": name,
            "category": "팝업스토어",  # 지역검색에서 못 찾은 경우도 이 파이프라인을 거친 이상 팝업으로 확정
            "address": extracted_address,
            "road_address": None,
            "lat": None,
            "lng": None,
            "link": row.get("link"),
            "phone": None,
            "business_hours": None,
            "menu": None,
            "tags": [],
            "area": None,
            "event_start": None,
            "event_end": None,
            "source": "blog_popup",
            "raw_json": json.dumps(row.to_dict(), ensure_ascii=False, default=str),
            "fetched_at": datetime.now(timezone.utc).isoformat(),
        }

    if not place.get("area"):
        place["area"] = extract_dong(place.get("address"))

    place["event_start"] = event_start.isoformat() if event_start else None
    place["event_end"] = event_end.isoformat() if event_end else None
    blog_place_rows.append(place)

df_blog_places = pd.DataFrame(blog_place_rows)
enriched_count = df_blog_places["lat"].notna().sum()
period_count = df_blog_places["event_end"].notna().sum()
area_count = df_blog_places["area"].notna().sum()
not_popup_category = (~df_blog_places["category"].str.startswith("팝업스토어", na=False)).sum()
print(f"연예뉴스성 글 제외: {skipped_news}건")
print(f"블로그 -> {len(df_blog_places)}건 매핑 (지역검색 보강: {enriched_count}건, 진행기간 파싱: {period_count}건, 동 파악: {area_count}건)")
print(f"category가 '팝업스토어'로 시작하지 않는 행(있으면 버그): {not_popup_category}건")
df_blog_places.head()

연예뉴스성 글 제외: 6건
블로그 -> 94건 매핑 (지역검색 보강: 37건, 진행기간 파싱: 38건, 동 파악: 7건)
category가 '팝업스토어'로 시작하지 않는 행(있으면 버그): 0건


,id,name,category,address,road_address,lat,lng,link,phone,business_hours,menu,tags,area,event_start,event_end,source,raw_json,fetched_at
0,f8820e18-5dce-5134-93f0-a7e085847600,메이크프렘 연남 팝업스토어,팝업스토어,서울특별시 마포구 연남동 390-34 1층,서울특별시 마포구 동교로38길 29 1층,37.561607,126.925679,https://www.instagram.com/makeprem,,None,None,[],연남동,2026-09-10,2026-09-13,blog_popup,"{""title"": ""<b>메이크프렘 연남 팝업스토어</b>"", ""link"": ""ht...",2026-09-11T04:37:24.280533+00:00
1,f8820e18-5dce-5134-93f0-a7e085847600,메이크프렘 연남 팝업스토어,팝업스토어,서울특별시 마포구 연남동 390-34 1층,서울특별시 마포구 동교로38길 29 1층,37.561607,126.925679,https://www.instagram.com/makeprem,,None,None,[],연남동,NaN,NaN,blog_popup,"{""title"": ""<b>메이크프렘 연남 팝업스토어</b>"", ""link"": ""ht...",2026-09-11T04:37:24.598659+00:00
2,68158aba-57b6-5847-89ae-cf99386d7819,홍대 꾸감 더 현대 대구 팝업 스토어,팝업스토어,서울 마포구 서교동 일대에서,NaN,NaN,NaN,https://blog.naver.com/ggugam1/224405873675,NaN,None,None,[],서교동,NaN,NaN,blog_popup,"{""title"": ""더현대대구에 상륙한 홍대 꾸감 팝업스토어 예고"", ""link"":...",2026-09-11T04:37:24.937226+00:00
3,5d24b188-2283-5752-b3c6-0ebfe97b9c6b,매력 팝업,팝업스토어,서울 마포구 양화로 162 좋은사람들빌딩 1~3층,NaN,NaN,NaN,https://blog.naver.com/sweetmayoyo/224405317231,NaN,None,None,[],NaN,NaN,NaN,blog_popup,"{""title"": ""폼폼푸린 팝업 카카오프렌즈 홍대플래그십 스토어 아이랑 옛날옛춘....",2026-09-11T04:37:25.247898+00:00
4,b84c8b63-0692-5858-a72b-c903fda69b8c,홍대 AK플라자 이누야샤 팝업 스토어,팝업스토어,"서울 마포구 양화로 188 (동교동, 애경타워) AK PLAZA 홍대 4층 LIMI",NaN,NaN,NaN,https://blog.naver.com/nabong_zip/224403819771,NaN,None,None,[],NaN,2026-09-01,2026-09-22,blog_popup,"{""title"": ""홍대 AK플라자 이누야샤 팝업스토어, 사전예약 방문 후기"", ""...",2026-09-11T04:37:25.542740+00:00


## 3. 병합 + 기간 만료 처리 + link 통일
지역검색 매핑 결과 + 블로그 매핑 결과를 합치고 `name`+`address` 기준 중복 제거.

**중요**: 이 노트북은 `places_mapo.csv`를 그때그때 원본(`naver_local_search_mapo.csv`, `naver_blog_content.csv`)에서 매번 처음부터 다시 계산해서 통째로 덮어씀 — 이전 출력과 병합하지 않음.
추출/정제 로직(정규식, 형태소 분석 등)을 고칠 때마다 예전 결과가 새 이름으로 남아 같은 장소가 두 줄로 쌓이는 문제가 있었어서, 원본 CSV가 유일한 소스가 되도록 바꿈. 재실행할 때마다 최신 추출 로직 기준으로 전체가 다시 계산됨.

`link`는 원본 그대로 두면 업체 홈페이지/인스타/블로그 글 주소 등 제각각이라(블로그 유래 장소는 특히 블로그 글 링크가 그대로 남음),
`name` 기준 네이버 검색 링크로 통일함 — 항상 존재하고, 클릭하면 그 장소를 검색해줌.

그 다음 `event_end`가 파싱되어 있고 오늘보다 과거인 행만 걸러서 `places_expired.csv`로 옮기고, 나머지만 `places_mapo.csv`에 저장.
`event_end`를 못 찾은 행(대부분의 상시 영업 장소 + 기간 파싱 실패한 팝업)은 만료 판정 없이 계속 활성으로 남음.

In [ ]:
from urllib.parse import quote

CSV_PATH = "data/places_mapo.csv"
EXPIRED_CSV_PATH = "data/places_expired.csv"

# 이거 일단 홀딩
# def build_naver_search_link(name):
#     """블로그 글 주소 등 제각각인 link 대신, 장소명으로 검색되는 네이버 검색 링크로 통일"""
#     return f"https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query={quote(str(name))}"


# 원본(지역검색/블로그) CSV가 유일한 소스 -> 매번 전체를 새로 계산해서 덮어씀 (이전 출력과 병합 안 함)
df_all = pd.concat([df_local_places, df_blog_places], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["name", "address"], keep="first").reset_index(drop=True)

# df_all["link"] = df_all["name"].apply(build_naver_search_link)

today = date.today().isoformat()
expired_mask = df_all["event_end"].notna() & (df_all["event_end"].astype(str) < today)

df_expired = df_all[expired_mask].reset_index(drop=True)
df_active = df_all[~expired_mask].reset_index(drop=True)

df_active.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
df_expired.to_csv(EXPIRED_CSV_PATH, index=False, encoding="utf-8-sig")

popup_active = (df_active["source"] == "blog_popup").sum()
popup_not_categorized = ((df_active["source"] == "blog_popup") & ~df_active["category"].str.startswith("팝업스토어", na=False)).sum()
print(f"활성 {len(df_active)}건 저장 -> {CSV_PATH} (팝업 파이프라인 유래: {popup_active}건, 그 중 카테고리 미확정: {popup_not_categorized}건)")
print(f"기간 만료 {len(df_expired)}건 보관 -> {EXPIRED_CSV_PATH}")
df_active.head()

활성 1229건 저장 -> data/places_mapo.csv (팝업 파이프라인 유래: 50건, 그 중 카테고리 미확정: 0건)
기간 만료 20건 보관 -> data/places_expired.csv


,id,name,category,address,road_address,lat,lng,link,phone,business_hours,menu,tags,area,event_start,event_end,source,raw_json,fetched_at
0,2fc349fc-7818-58b2-8dbe-c93d58c803a4,슈가코인노래연습장 공덕점,오락시설>노래방,서울특별시 마포구 공덕동 105-159 지하 1층,서울특별시 마포구 마포대로 180 지하 1층,37.550182,126.955507,https://search.naver.com/search.naver?where=ne...,None,None,None,[],공덕동,None,None,local,"{""title"": ""슈가코인노래연습장 공덕점"", ""link"": ""http://m.f...",2026-09-11T04:37:19.792018+00:00
1,546b5896-cbb6-59d9-8a03-b2e96cc5a778,뮤즈코인노래연습장,오락시설>노래방,서울특별시 마포구 신공덕동 20-18 지하1층,서울특별시 마포구 백범로37길 22 지하1층,37.544066,126.954814,https://search.naver.com/search.naver?where=ne...,None,None,None,[],공덕동,None,None,local,"{""title"": ""뮤즈코인노래연습장"", ""link"": NaN, ""category""...",2026-09-11T04:37:19.792318+00:00
2,cdc94e5c-cea2-55fc-92e9-e4fb93385f36,클럽스트라이크 볼링장,"스포츠,오락>볼링장",서울특별시 마포구 도화동 36 4층 클럽스트라이크 볼링장,서울특별시 마포구 마포대로 52 4층 클럽스트라이크 볼링장,37.540328,126.947665,https://search.naver.com/search.naver?where=ne...,None,None,None,[],공덕동,None,None,local,"{""title"": ""클럽스트라이크 <b>볼링장</b>"", ""link"": ""https...",2026-09-11T04:37:19.792960+00:00
3,931ad374-feb2-5dff-b639-43f7a71319eb,뉴청룡볼링장,"스포츠,오락>볼링장",서울특별시 용산구 갈월동 98-38 청룡빌딩 지하1층,서울특별시 용산구 한강대로 257 청룡빌딩 지하1층,37.541356,126.972639,https://search.naver.com/search.naver?where=ne...,None,None,None,[],공덕동,None,None,local,"{""title"": ""뉴청룡<b>볼링장</b>"", ""link"": NaN, ""categ...",2026-09-11T04:37:19.793230+00:00
4,6160b3c7-10c8-5bd8-9a97-76df97e45a61,타겟볼링,"스포츠,오락>볼링장",서울특별시 서대문구 대현동 101-7 혜우빌딩 지하1층,서울특별시 서대문구 신촌역로 10 혜우빌딩 지하1층,37.557589,126.943072,https://search.naver.com/search.naver?where=ne...,None,None,None,[],공덕동,None,None,local,"{""title"": ""타겟볼링"", ""link"": NaN, ""category"": ""스포...",2026-09-11T04:37:19.793412+00:00
